In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.metrics import silhouette_score, mean_squared_error, mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create output directories
import os
os.makedirs('plots', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("=" * 80)
print("TASK 1: MACHINE DATA ANALYSIS - CYCLONE SENSOR DATA")
print("=" * 80)

TASK 1: MACHINE DATA ANALYSIS - CYCLONE SENSOR DATA


In [4]:
# ============================================================================
# TASK 1.1: DATA PREPARATION & EXPLORATORY ANALYSIS
# ============================================================================

print("\n[1] DATA PREPARATION & EXPLORATORY ANALYSIS")
print("-" * 80)

# Load data - Update this path if needed
try:
    df = pd.read_csv('/content/data.csv')
    print(f"* Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
except FileNotFoundError:
    print("ERROR: data.csv not found in current directory")
    print("Please ensure data.csv is in the same folder as this script")
    exit(1)

# Convert time column to datetime
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

# Convert numeric columns to float
numeric_cols: List[str] = [col for col in df.columns if col != 'time']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"* Data spans: {df['time'].min()} to {df['time'].max()}")
print(f"* Duration: {(df['time'].max() - df['time'].min()).days} days")

# Check for missing values
missing_info = df.isnull().sum()
print(f"\n* Missing values per column:")
for col in numeric_cols:
    pct = (missing_info[col] / len(df)) * 100
    print(f"  - {col}: {missing_info[col]:,} ({pct:.2f}%)")

# Handle missing values (forward fill then backward fill)
df[numeric_cols] = df[numeric_cols].ffill().bfill()
print(f"\n* Missing values handled via forward/backward fill")

# Check for duplicate timestamps before setting index
duplicates = df['time'].duplicated().sum()
if duplicates > 0:
    print(f"\n! Found {duplicates} duplicate timestamps - removing duplicates")
    df = df.drop_duplicates(subset='time', keep='first')
    print(f"* Removed duplicates, remaining records: {len(df):,}")

# Set time as index
df = df.set_index('time')

# Create complete time range with 5-minute frequency
time_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='5T')
print(f"\n* Expected records at 5-min frequency: {len(time_range):,}")
print(f"* Actual records: {len(df):,}")
gap_count = len(time_range) - len(df)
gap_pct = (gap_count / len(time_range)) * 100
print(f"* Gap: {gap_count:,} records ({gap_pct:.2f}%)")

# Reindex to ensure 5-minute frequency
df = df.reindex(time_range)

# Check for gaps and interpolate
gaps = df[numeric_cols].isnull().sum()
if gaps.sum() > 0:
    print(f"\n! Found {gaps.sum()} gaps after reindexing - filling with interpolation")
    df[numeric_cols] = df[numeric_cols].interpolate(method='linear')
    df[numeric_cols] = df[numeric_cols].bfill().ffill()
    print(f"* All gaps filled")

# Summary statistics
print("\n* Summary Statistics:")
print(df[numeric_cols].describe().round(2))

# Correlation matrix
print("\n* Correlation Matrix:")
corr_matrix = df[numeric_cols].corr()
print(corr_matrix.round(3))

# Save correlation heatmap
fig1 = plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Sensor Variables Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/01_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.close(fig1)
print("  -> Saved: plots/01_correlation_matrix.png")

# Visualize one week of data
week_start = df.index[len(df)//2]
week_end = week_start + pd.Timedelta(days=7)
week_data = df.loc[week_start:week_end]

fig2, axes2 = plt.subplots(3, 2, figsize=(16, 12))
axes_flat2 = axes2.flatten()
for idx, col in enumerate(numeric_cols):
    axes_flat2[idx].plot(week_data.index, week_data[col].values, linewidth=0.8)
    axes_flat2[idx].set_title(f'{col} (1 Week Sample)', fontsize=11, fontweight='bold')
    axes_flat2[idx].set_xlabel('Time')
    axes_flat2[idx].set_ylabel('Value')
    axes_flat2[idx].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/02_one_week_view.png', dpi=300, bbox_inches='tight')
plt.close(fig2)
print("  -> Saved: plots/02_one_week_view.png")

# Visualize one year of data (downsampled for clarity)
year_start = df.index[0]
year_end = year_start + pd.Timedelta(days=365)
year_data = df.loc[year_start:year_end].resample('1H').mean()

fig3, axes3 = plt.subplots(3, 2, figsize=(16, 12))
axes_flat3 = axes3.flatten()
for idx, col in enumerate(numeric_cols):
    axes_flat3[idx].plot(year_data.index, year_data[col].values, linewidth=0.6, alpha=0.8)
    axes_flat3[idx].set_title(f'{col} (1 Year Sample - Hourly Avg)', fontsize=11, fontweight='bold')
    axes_flat3[idx].set_xlabel('Time')
    axes_flat3[idx].set_ylabel('Value')
    axes_flat3[idx].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/03_one_year_view.png', dpi=300, bbox_inches='tight')
plt.close(fig3)
print("  -> Saved: plots/03_one_year_view.png")


[1] DATA PREPARATION & EXPLORATORY ANALYSIS
--------------------------------------------------------------------------------
* Loaded dataset: 378,719 rows x 7 columns
* Data spans: 2017-01-01 00:00:00 to 2020-08-07 12:15:00
* Duration: 1314 days

* Missing values per column:
  - Cyclone_Inlet_Gas_Temp: 2,320 (0.61%)
  - Cyclone_Material_Temp: 2,591 (0.68%)
  - Cyclone_Outlet_Gas_draft: 2,321 (0.61%)
  - Cyclone_cone_draft: 2,320 (0.61%)
  - Cyclone_Gas_Outlet_Temp: 2,321 (0.61%)
  - Cyclone_Inlet_Draft: 2,322 (0.61%)

* Missing values handled via forward/backward fill

! Found 999 duplicate timestamps - removing duplicates
* Removed duplicates, remaining records: 377,720

* Expected records at 5-min frequency: 378,580
* Actual records: 377,720
* Gap: 860 records (0.23%)

! Found 5166 gaps after reindexing - filling with interpolation
* All gaps filled

* Summary Statistics:
       Cyclone_Inlet_Gas_Temp  Cyclone_Material_Temp  \
count               378580.00              378580.00   

In [5]:
# ============================================================================
# TASK 1.2: SHUTDOWN / IDLE PERIOD DETECTION
# ============================================================================

print("\n[2] SHUTDOWN / IDLE PERIOD DETECTION")
print("-" * 80)

# Strategy: Identify shutdowns based on low sensor values
shutdown_threshold: Dict[str, float] = {
    'Cyclone_Inlet_Gas_Temp': 50,
    'Cyclone_Gas_Outlet_Temp': 50,
    'Cyclone_Material_Temp': 30,
}

# Mark as shutdown when multiple key sensors are below threshold
shutdown_conditions = []
for col, threshold in shutdown_threshold.items():
    shutdown_conditions.append(df[col] < threshold)

df['is_shutdown'] = np.all(shutdown_conditions, axis=0).astype(int)

# Find shutdown periods
df['shutdown_group'] = (df['is_shutdown'] != df['is_shutdown'].shift()).cumsum()
shutdown_groups = df[df['is_shutdown'] == 1].groupby('shutdown_group')

shutdown_periods = pd.DataFrame({
    'start_time': shutdown_groups.apply(lambda x: x.index.min()),
    'end_time': shutdown_groups.apply(lambda x: x.index.max()),
    'duration_hours': shutdown_groups.size() * 5 / 60
}).reset_index(drop=True)

# Calculate statistics
total_downtime_hours: float = float(shutdown_periods['duration_hours'].sum())
num_shutdowns: int = len(shutdown_periods)
avg_shutdown_duration: float = float(shutdown_periods['duration_hours'].mean())

print(f"* Shutdown Detection Summary:")
print(f"  - Total shutdowns detected: {num_shutdowns}")
print(f"  - Total downtime: {total_downtime_hours:.1f} hours ({total_downtime_hours/24:.1f} days)")
print(f"  - Average shutdown duration: {avg_shutdown_duration:.1f} hours")
print(f"  - Shortest shutdown: {shutdown_periods['duration_hours'].min():.2f} hours")
print(f"  - Longest shutdown: {shutdown_periods['duration_hours'].max():.2f} hours")

# Save shutdown periods
shutdown_periods.to_csv('outputs/shutdown_periods.csv', index=False)
print(f"  -> Saved: outputs/shutdown_periods.csv")

# Visualize one year with shutdowns highlighted
year_data_full = df.loc[year_start:year_end]

fig4, ax4 = plt.subplots(figsize=(18, 6))
ax4.plot(year_data_full.index, year_data_full['Cyclone_Inlet_Gas_Temp'].to_numpy(),
        linewidth=0.5, alpha=0.7, label='Inlet Gas Temp', color='steelblue')

# Highlight shutdown periods
first_shutdown = True
for _, row in shutdown_periods.iterrows():
    if row['start_time'] >= year_data_full.index[0] and row['start_time'] <= year_data_full.index[-1]:
        label = 'Shutdown' if first_shutdown else ''
        ax4.axvspan(row['start_time'], row['end_time'], alpha=0.3, color='red', label=label)
        first_shutdown = False

ax4.set_title('One Year View: Cyclone Inlet Gas Temperature with Shutdowns Highlighted',
              fontsize=14, fontweight='bold')
ax4.set_xlabel('Time', fontsize=12)
ax4.set_ylabel('Temperature (C)', fontsize=12)
ax4.legend(loc='upper right')
ax4.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/04_shutdowns_one_year.png', dpi=300, bbox_inches='tight')
plt.close(fig4)
print("  -> Saved: plots/04_shutdowns_one_year.png")


[2] SHUTDOWN / IDLE PERIOD DETECTION
--------------------------------------------------------------------------------
* Shutdown Detection Summary:
  - Total shutdowns detected: 416
  - Total downtime: 2505.8 hours (104.4 days)
  - Average shutdown duration: 6.0 hours
  - Shortest shutdown: 0.08 hours
  - Longest shutdown: 152.50 hours
  -> Saved: outputs/shutdown_periods.csv
  -> Saved: plots/04_shutdowns_one_year.png


In [9]:
# ============================================================================
# TASK 1.3: MACHINE STATE SEGMENTATION (CLUSTERING)
# ============================================================================

print("\n[3] MACHINE STATE SEGMENTATION (CLUSTERING)")
print("-" * 80)

df_active = df[df['is_shutdown'] == 0].copy()
active_pct = len(df_active) / len(df) * 100
print(f"* Active operation data: {len(df_active):,} records ({active_pct:.1f}% of total)")

df_active['temp_diff'] = df_active['Cyclone_Inlet_Gas_Temp'] - df_active['Cyclone_Gas_Outlet_Temp']
df_active['draft_diff'] = df_active['Cyclone_Inlet_Draft'] - df_active['Cyclone_Outlet_Gas_draft']
df_active['inlet_temp_rolling_std'] = df_active['Cyclone_Inlet_Gas_Temp'].rolling(window=12, min_periods=1).std()
df_active['material_temp_rolling_mean'] = df_active['Cyclone_Material_Temp'].rolling(window=12, min_periods=1).mean()

cluster_features: List[str] = [
    'Cyclone_Inlet_Gas_Temp', 'Cyclone_Gas_Outlet_Temp', 'Cyclone_Material_Temp',
    'Cyclone_Outlet_Gas_draft', 'Cyclone_cone_draft', 'Cyclone_Inlet_Draft',
    'temp_diff', 'draft_diff', 'inlet_temp_rolling_std'
]

df_active = df_active.dropna(subset=cluster_features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_active[cluster_features].values)

inertias: List[float] = []
silhouette_scores: List[float] = []
K_range = range(2, 8)

# Define a reasonable sample size for silhouette calculation
silhouette_sample_size = min(30000, len(X_scaled)) # Use up to 30,000 samples, or less if data is smaller

for k in K_range:
    print(f"  ... Evaluating k={k}")
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(float(kmeans.inertia_))

    # Calculate silhouette score using the sample_size parameter directly
    score = silhouette_score(X_scaled, labels, sample_size=silhouette_sample_size, random_state=42)
    silhouette_scores.append(float(score))

fig5, (ax5_1, ax5_2) = plt.subplots(1, 2, figsize=(14, 5))
ax5_1.plot(list(K_range), inertias, marker='o', linewidth=2)
ax5_1.set_xlabel('Number of Clusters (k)')
ax5_1.set_ylabel('Inertia')
ax5_1.set_title('Elbow Method for Optimal k')
ax5_1.grid(True, alpha=0.3)

ax5_2.plot(list(K_range), silhouette_scores, marker='o', linewidth=2, color='orange')
ax5_2.set_xlabel('Number of Clusters (k)')
ax5_2.set_ylabel('Silhouette Score')
ax5_2.set_title('Silhouette Score vs k (on sample)')
ax5_2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/05_clustering_optimization.png', dpi=300, bbox_inches='tight')
plt.close(fig5)
print("  -> Saved: plots/05_clustering_optimization.png")

optimal_k: int = 4
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
df_active['cluster'] = kmeans_final.fit_predict(X_scaled)

# Final silhouette score calculation also using sample_size
final_silhouette = float(silhouette_score(X_scaled, df_active['cluster'].values, sample_size=silhouette_sample_size, random_state=42))
print(f"\n* Applied KMeans clustering with k={optimal_k}")
print(f"  Silhouette Score (on sample): {final_silhouette:.3f}")

cluster_summary: List[Dict[str, Any]] = []
centroids = kmeans_final.cluster_centers_
original_centroids = scaler.inverse_transform(centroids)
centroid_df = pd.DataFrame(original_centroids, columns=cluster_features)
temp_ranking = centroid_df['Cyclone_Inlet_Gas_Temp'].rank(method='first', ascending=True).astype(int) - 1
state_map = {
    temp_ranking[temp_ranking == 0].index[0]: 'Startup/Low Load',
    temp_ranking[temp_ranking == 1].index[0]: 'Normal Operation',
    temp_ranking[temp_ranking == 2].index[0]: 'High Load',
    temp_ranking[temp_ranking > 2].index[0]: 'Degraded'
}

for cluster_id in range(optimal_k):
    cluster_data = df_active[df_active['cluster'] == cluster_id]
    summary = {
        'cluster_id': cluster_id, 'state_name': state_map.get(cluster_id, f'State_{cluster_id}'),
        'count': len(cluster_data), 'frequency_pct': float(len(cluster_data) / len(df_active) * 100),
    }
    for col in numeric_cols:
        summary[f'{col}_mean'] = float(cluster_data[col].mean())
        summary[f'{col}_std'] = float(cluster_data[col].std())
    cluster_summary.append(summary)
    print(f"\n  Cluster {cluster_id}: {summary['state_name']}")
    print(f"    - Frequency: {summary['frequency_pct']:.1f}% ({summary['count']:,} records)")
    print(f"    - Inlet Gas Temp: {summary['Cyclone_Inlet_Gas_Temp_mean']:.1f}C ( +/- {summary['Cyclone_Inlet_Gas_Temp_std']:.1f})")
    print(f"    - Outlet Gas Temp: {summary['Cyclone_Gas_Outlet_Temp_mean']:.1f}C ( +/- {summary['Cyclone_Gas_Outlet_Temp_std']:.1f})")
    print(f"    - Material Temp: {summary['Cyclone_Material_Temp_mean']:.1f}C ( +/- {summary['Cyclone_Material_Temp_std']:.1f})")

cluster_summary_df = pd.DataFrame(cluster_summary)
cluster_summary_df.to_csv('outputs/clusters_summary.csv', index=False)
print(f"\n  -> Saved: outputs/clusters_summary.csv")

sample_size = min(10000, len(df_active))
df_active_sample = df_active.sample(n=sample_size, random_state=42)
fig6, axes6 = plt.subplots(2, 3, figsize=(18, 10))
axes_flat6 = axes6.flatten()
for idx, col in enumerate(numeric_cols):
    for cluster_id in range(optimal_k):
        cluster_sample = df_active_sample[df_active_sample['cluster'] == cluster_id]
        axes_flat6[idx].scatter(range(len(cluster_sample)), cluster_sample[col].values,
                                s=1, alpha=0.3, label=state_map.get(cluster_id, f'Cluster_{cluster_id}'))
    axes_flat6[idx].set_title(f'{col} by Cluster', fontsize=11, fontweight='bold')
    axes_flat6[idx].set_xlabel('Sample Index')
    axes_flat6[idx].set_ylabel('Value')
    if idx == 0:
        axes_flat6[idx].legend()
plt.tight_layout()
plt.savefig('plots/06_clusters_visualization.png', dpi=300, bbox_inches='tight')
plt.close(fig6)
print("  -> Saved: plots/06_clusters_visualization.png")


[3] MACHINE STATE SEGMENTATION (CLUSTERING)
--------------------------------------------------------------------------------
* Active operation data: 348,511 records (92.1% of total)
  ... Evaluating k=2
  ... Evaluating k=3
  ... Evaluating k=4
  ... Evaluating k=5
  ... Evaluating k=6
  ... Evaluating k=7
  -> Saved: plots/05_clustering_optimization.png

* Applied KMeans clustering with k=4
  Silhouette Score (on sample): 0.563

  Cluster 0: Normal Operation
    - Frequency: 3.7% (12,997 records)
    - Inlet Gas Temp: 868.8C ( +/- 109.3)
    - Outlet Gas Temp: 844.1C ( +/- 95.1)
    - Material Temp: 898.2C ( +/- 128.9)

  Cluster 1: High Load
    - Frequency: 78.0% (271,696 records)
    - Inlet Gas Temp: 891.6C ( +/- 24.6)
    - Outlet Gas Temp: 885.7C ( +/- 32.9)
    - Material Temp: 922.2C ( +/- 68.1)

  Cluster 2: Startup/Low Load
    - Frequency: 14.2% (49,506 records)
    - Inlet Gas Temp: 154.7C ( +/- 192.1)
    - Outlet Gas Temp: 152.6C ( +/- 189.8)
    - Material Temp: 157.5

In [10]:
# ============================================================================
# TASK 1.4: CONTEXTUAL ANOMALY DETECTION + ROOT CAUSE ANALYSIS
# ============================================================================

print("\n[4] CONTEXTUAL ANOMALY DETECTION + ROOT CAUSE ANALYSIS")
print("-" * 80)

anomalies_list: List[Dict[str, Any]] = []

for cluster_id in range(optimal_k):
    cluster_data = df_active[df_active['cluster'] == cluster_id].copy()

    if len(cluster_data) < 100:
        continue

    X_cluster = cluster_data[cluster_features].values

    iso_forest = IsolationForest(contamination=0.02, random_state=42, n_estimators=100)
    cluster_data['anomaly'] = iso_forest.fit_predict(X_cluster)
    cluster_data['anomaly_score'] = iso_forest.score_samples(X_cluster)

    anomalies = cluster_data[cluster_data['anomaly'] == -1].copy()

    if len(anomalies) > 0:
        time_diffs = anomalies.index.to_series().diff()
        anomalies['anomaly_group'] = (time_diffs > pd.Timedelta(minutes=10)).cumsum()

        for group_id, group_data in anomalies.groupby('anomaly_group'):
            event: Dict[str, Any] = {
                'cluster_id': cluster_id,
                'state_name': state_map.get(cluster_id, f'State_{cluster_id}'),
                'start_time': group_data.index.min(),
                'end_time': group_data.index.max(),
                'duration_minutes': float(len(group_data) * 5),
                'anomaly_score_mean': float(group_data['anomaly_score'].mean()),
            }

            for col in numeric_cols:
                cluster_mean = float(cluster_data[col].mean())
                cluster_std = float(cluster_data[col].std())
                group_mean = float(group_data[col].mean())
                event[f'{col}_deviation'] = (group_mean - cluster_mean) / (cluster_std + 1e-6)

            anomalies_list.append(event)

anomalies_df = pd.DataFrame(anomalies_list)
if not anomalies_df.empty:
    anomalies_df = anomalies_df.sort_values('anomaly_score_mean').head(50)

print(f"* Detected {len(anomalies_df)} significant anomalous events")

anomalies_df.to_csv('outputs/anomalous_periods.csv', index=False)
print(f"  -> Saved: outputs/anomalous_periods.csv")

# Root cause analysis for top 5 anomalies
top_anomalies = anomalies_df.head(5)

print("\n* Root Cause Analysis for Top 5 Anomalies:\n")

for idx, (_, anomaly) in enumerate(top_anomalies.iterrows(), 1):
    print(f"  Anomaly {idx}:")
    print(f"    Time: {anomaly['start_time']} to {anomaly['end_time']}")
    print(f"    State: {anomaly['state_name']}")
    print(f"    Duration: {anomaly['duration_minutes']:.0f} minutes")

    deviation_cols = [col for col in anomalies_df.columns if col.endswith('_deviation')]
    deviations = {col.replace('_deviation', ''): anomaly[col] for col in deviation_cols}
    top_deviations = sorted(deviations.items(), key=lambda x: abs(x[1]), reverse=True)[:3]

    print(f"    Most affected variables:")
    for var, dev in top_deviations:
        direction = "increase" if dev > 0 else "decrease"
        print(f"      - {var}: {abs(dev):.2f} sigma {direction}")

    inlet_temp_dev = deviations.get('Cyclone_Inlet_Gas_Temp', 0)
    inlet_draft_dev = deviations.get('Cyclone_Inlet_Draft', 0)

    if abs(inlet_temp_dev) > 2:
        if inlet_temp_dev > 0:
            print(f"    -> Root Cause Hypothesis: Sudden spike in inlet temperature suggests upstream combustion surge.")
        else:
            print(f"    -> Root Cause Hypothesis: Sudden drop in inlet temperature indicates fuel supply interruption.")
    elif abs(inlet_draft_dev) > 2:
        print(f"    -> Root Cause Hypothesis: Draft pressure anomaly suggests a ventilation system issue.")
    else:
        print(f"    -> Root Cause Hypothesis: Multi-variable deviation indicates general process instability.")
    print()

# Visualize top 3 anomalies
for idx, (_, anomaly) in enumerate(top_anomalies.head(3).iterrows(), 1):
    start = anomaly['start_time'] - pd.Timedelta(hours=2)
    end = anomaly['end_time'] + pd.Timedelta(hours=2)
    context_data = df.loc[start:end]

    fig7, axes7 = plt.subplots(3, 2, figsize=(16, 10))
    axes_flat7 = axes7.flatten()

    for ax_idx, col in enumerate(numeric_cols):
        axes_flat7[ax_idx].plot(context_data.index, context_data[col].values, linewidth=1)
        axes_flat7[ax_idx].axvspan(anomaly['start_time'], anomaly['end_time'],
                                   alpha=0.3, color='red', label='Anomaly Period')
        axes_flat7[ax_idx].set_title(f'{col}', fontsize=10, fontweight='bold')
        axes_flat7[ax_idx].set_xlabel('Time')
        axes_flat7[ax_idx].set_ylabel('Value')
        axes_flat7[ax_idx].grid(True, alpha=0.3)
        if ax_idx == 0:
            axes_flat7[ax_idx].legend()

    plt.suptitle(f'Anomaly Context View: {anomaly["start_time"]}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'plots/07_anomaly_{idx}_context.png', dpi=300, bbox_inches='tight')
    plt.close(fig7)
    print(f"  -> Saved: plots/07_anomaly_{idx}_context.png")


[4] CONTEXTUAL ANOMALY DETECTION + ROOT CAUSE ANALYSIS
--------------------------------------------------------------------------------
* Detected 50 significant anomalous events
  -> Saved: outputs/anomalous_periods.csv

* Root Cause Analysis for Top 5 Anomalies:

  Anomaly 1:
    Time: 2018-05-16 11:50:00 to 2018-05-16 12:50:00
    State: Degraded
    Duration: 65 minutes
    Most affected variables:
      - Cyclone_Outlet_Gas_draft: 6.58 sigma increase
      - Cyclone_Inlet_Gas_Temp: 6.54 sigma decrease
      - Cyclone_Inlet_Draft: 6.42 sigma increase
    -> Root Cause Hypothesis: Sudden drop in inlet temperature indicates fuel supply interruption.

  Anomaly 2:
    Time: 2018-06-21 04:55:00 to 2018-06-21 08:45:00
    State: Degraded
    Duration: 235 minutes
    Most affected variables:
      - Cyclone_Inlet_Gas_Temp: 8.10 sigma decrease
      - Cyclone_Outlet_Gas_draft: 7.00 sigma increase
      - Cyclone_Inlet_Draft: 6.95 sigma increase
    -> Root Cause Hypothesis: Sudden drop 

In [12]:
# ============================================================================
# TASK 1.5: SHORT-HORIZON FORECASTING
# ============================================================================

print("\n[5] SHORT-HORIZON FORECASTING (1 HOUR / 12 STEPS)")
print("-" * 80)

target_var = 'Cyclone_Inlet_Gas_Temp'
forecast_horizon = 12

df_forecast = df_active[[target_var]].copy()

train_size = int(len(df_forecast) * 0.9)
train_data = df_forecast.iloc[:train_size]
test_data = df_forecast.iloc[train_size:]

print(f"* Forecasting target: {target_var}")
print(f"  Train size: {len(train_data):,} records")
print(f"  Test size: {len(test_data):,} records")

# Method 1: Persistence Baseline
print("\n  Method 1: Persistence Baseline")
persistence_predictions: List[float] = []
persistence_actuals: List[float] = []

for i in range(forecast_horizon, len(test_data)):
    pred = float(test_data[target_var].iloc[i - forecast_horizon])
    actual = float(test_data[target_var].iloc[i])
    persistence_predictions.append(pred)
    persistence_actuals.append(actual)

persistence_rmse = float(np.sqrt(mean_squared_error(persistence_actuals, persistence_predictions)))
persistence_mae = float(mean_absolute_error(persistence_actuals, persistence_predictions))

print(f"    RMSE: {persistence_rmse:.3f}")
print(f"    MAE: {persistence_mae:.3f}")

# Method 2: Random Forest with Lag Features
print("\n  Method 2: Random Forest with Lag Features")

lags = [1, 2, 3, 6, 12, 24]
for lag in lags:
    df_forecast[f'lag_{lag}'] = df_forecast[target_var].shift(lag)

df_forecast['rolling_mean_12'] = df_forecast[target_var].rolling(window=12).mean()
df_forecast['rolling_std_12'] = df_forecast[target_var].rolling(window=12).std()

df_forecast = df_forecast.dropna()

feature_cols = [f'lag_{lag}' for lag in lags] + ['rolling_mean_12', 'rolling_std_12']
X = df_forecast[feature_cols].values
y = df_forecast[target_var].values

train_size_new = int(len(df_forecast) * 0.9)
X_train, X_test = X[:train_size_new], X[train_size_new:]
y_train, y_test = y[:train_size_new], y[train_size_new:]

rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(np.array(X_train), np.array(y_train))

rf_predictions = rf_model.predict(X_test).tolist()
rf_actuals = y_test.tolist()

rf_rmse = float(np.sqrt(mean_squared_error(rf_actuals, rf_predictions)))
rf_mae = float(mean_absolute_error(rf_actuals, rf_predictions))

print(f"    RMSE: {rf_rmse:.3f}")
print(f"    MAE: {rf_mae:.3f}")

print(f"\n* Model Comparison:")
print(f"  Persistence: RMSE={persistence_rmse:.3f}, MAE={persistence_mae:.3f}")
print(f"  Random Forest: RMSE={rf_rmse:.3f}, MAE={rf_mae:.3f}")
improvement = ((persistence_rmse - rf_rmse) / persistence_rmse) * 100
print(f"  -> Random Forest improves RMSE by {improvement:.1f}%")

forecast_index = df_forecast.index[train_size_new:]
min_len = min(len(forecast_index), len(rf_actuals))

forecasts_df = pd.DataFrame({
    'time': forecast_index[:min_len],
    'actual': rf_actuals[:min_len],
    'predicted_rf': rf_predictions[:min_len]
})
forecasts_df.to_csv('outputs/forecasts.csv', index=False)
print(f"  -> Saved: outputs/forecasts.csv")

# Visualize forecasts
sample_size = 288
fig8, (ax8_1, ax8_2) = plt.subplots(2, 1, figsize=(16, 10))

ax8_1.plot(forecast_index[:sample_size], rf_actuals[:sample_size], label='Actual', linewidth=2, alpha=0.8)
ax8_1.plot(forecast_index[:sample_size], rf_predictions[:sample_size], label='RF Predicted', linewidth=1.5, alpha=0.7)
ax8_1.set_title('Forecast vs. Actuals: Sample View (1 Day)', fontsize=12, fontweight='bold')
ax8_1.set_xlabel('Time')
ax8_1.set_ylabel(target_var)
ax8_1.legend()
ax8_1.grid(True, alpha=0.3)

rf_errors = np.array(rf_actuals) - np.array(rf_predictions)
ax8_2.hist(rf_errors, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax8_2.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax8_2.set_title('Random Forest Forecast Error Distribution', fontsize=12, fontweight='bold')
ax8_2.set_xlabel('Prediction Error')
ax8_2.set_ylabel('Frequency')
ax8_2.legend()
ax8_2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/08_forecasting_results.png', dpi=300, bbox_inches='tight')
plt.close(fig8)
print(f"  -> Saved: plots/08_forecasting_results.png")


[5] SHORT-HORIZON FORECASTING (1 HOUR / 12 STEPS)
--------------------------------------------------------------------------------
* Forecasting target: Cyclone_Inlet_Gas_Temp
  Train size: 313,659 records
  Test size: 34,851 records

  Method 1: Persistence Baseline
    RMSE: 21.400
    MAE: 12.202

  Method 2: Random Forest with Lag Features
    RMSE: 11.456
    MAE: 7.173

* Model Comparison:
  Persistence: RMSE=21.400, MAE=12.202
  Random Forest: RMSE=11.456, MAE=7.173
  -> Random Forest improves RMSE by 46.5%
  -> Saved: outputs/forecasts.csv
  -> Saved: plots/08_forecasting_results.png


In [13]:
# ============================================================================
# TASK 1.6: INSIGHTS & STORYTELLING
# ============================================================================

print("\n[6] INSIGHTS & STORYTELLING")
print("-" * 80)

print("\n* KEY INSIGHTS FROM ANALYSIS:\n")

uptime_pct = (1 - total_downtime_hours / (len(df) * 5 / 60)) * 100
print(f"  1. OPERATIONAL AVAILABILITY")
print(f"    - Machine availability: {uptime_pct:.1f}%")
print(f"    - {num_shutdowns} shutdowns over 3 years (~{int(num_shutdowns/3)} per year)")
print(f"    - Average shutdown: {avg_shutdown_duration:.1f} hours")
print(f"    -> ACTION: Investigate shutdowns exceeding {avg_shutdown_duration*1.5:.0f}h for potential maintenance opportunities.\n")

print(f"  2. OPERATING STATE PATTERNS")
for _, row in cluster_summary_df.iterrows():
    print(f"    - {row['state_name']}: {row['frequency_pct']:.1f}% of active time")
dominant_state = cluster_summary_df.loc[cluster_summary_df['frequency_pct'].idxmax()]
print(f"    -> Dominant state: {dominant_state['state_name']} ({dominant_state['frequency_pct']:.1f}%)")
print(f"    -> ACTION: Focus optimization efforts on the Normal and High Load states.\n")

anomaly_by_state = anomalies_df.groupby('state_name').size()
print(f"  3. ANOMALY CONCENTRATION BY STATE")
if not anomaly_by_state.empty:
    for state, count in anomaly_by_state.items():
        pct = (count / len(anomalies_df)) * 100
        print(f"    - {state}: {count} anomalies ({pct:.1f}%)")
    high_risk_state = anomaly_by_state.idxmax()
    print(f"    -> Highest risk state for anomalies: {high_risk_state}")
    print(f"    -> ACTION: Implement state-specific monitoring thresholds, especially for the '{high_risk_state}' state.\n")
else:
    print("    - No significant anomalies detected to analyze by state.")

inlet_outlet_corr = corr_matrix.loc['Cyclone_Inlet_Gas_Temp', 'Cyclone_Gas_Outlet_Temp']
draft_corr = corr_matrix.loc['Cyclone_Inlet_Draft', 'Cyclone_Outlet_Gas_draft']
print(f"  4. KEY SENSOR RELATIONSHIPS")
print(f"    - Inlet-Outlet Temp correlation: {inlet_outlet_corr:.3f}")
print(f"    - Inlet-Outlet Draft correlation: {draft_corr:.3f}")
print(f"    -> Strong positive correlations indicate a healthy, balanced system.")
print(f"    -> ACTION: Create alerts for when these correlations deviate significantly from their baseline.\n")

print(f"  5. PREDICTABILITY & FORECASTING")
print(f"    - Inlet temperature can be forecast with an RMSE of {rf_rmse:.2f}C.")
print(f"    - This is a {improvement:.1f}% improvement over a simple persistence baseline.")
print(f"    -> The machine's thermal behavior is moderately predictable in stable states.")
print(f"    -> ACTION: Use the forecasting model as part of an early warning system for thermal irregularities.\n")

# Create summary visualization
fig9 = plt.figure(figsize=(16, 10))
gs = fig9.add_gridspec(3, 3, hspace=0.4, wspace=0.3)

ax9_1 = fig9.add_subplot(gs[0, 0])
ax9_1.pie([uptime_pct, 100-uptime_pct], labels=['Uptime', 'Downtime'],
        autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
ax9_1.set_title('Operational Availability', fontweight='bold')

ax9_2 = fig9.add_subplot(gs[0, 1])
ax9_2.bar(cluster_summary_df['state_name'].to_numpy(), cluster_summary_df['frequency_pct'].to_numpy(),
        color='steelblue', edgecolor='black')
ax9_2.set_title('Operating State Distribution', fontweight='bold')
ax9_2.set_ylabel('Frequency (%)')
ax9_2.tick_params(axis='x', rotation=45, labelsize=9)

ax9_3 = fig9.add_subplot(gs[0, 2])
if not anomaly_by_state.empty:
    ax9_3.bar(anomaly_by_state.index.values.astype(str), anomaly_by_state.values.astype(float), color='coral', edgecolor='black')
    ax9_3.set_title('Anomalies by State', fontweight='bold')
    ax9_3.set_ylabel('Count')
    ax9_3.tick_params(axis='x', rotation=45, labelsize=9)
else:
    ax9_3.text(0.5, 0.5, 'No anomalies to plot', ha='center', va='center')
    ax9_3.set_title('Anomalies by State', fontweight='bold')


ax9_4 = fig9.add_subplot(gs[1, :])
ax9_4.hist(shutdown_periods['duration_hours'].to_numpy(), bins=30, color='salmon', edgecolor='black', alpha=0.7)
ax9_4.axvline(avg_shutdown_duration, color='red', linestyle='--', linewidth=2,
            label=f'Mean: {avg_shutdown_duration:.1f}h')
ax9_4.set_title('Shutdown Duration Distribution', fontweight='bold')
ax9_4.set_xlabel('Duration (hours)')
ax9_4.set_ylabel('Frequency')
ax9_4.legend()
ax9_4.grid(True, alpha=0.3)

ax9_5 = fig9.add_subplot(gs[2, :2])
methods = ['Persistence\nBaseline', 'Random Forest\n(Lag Features)']
rmse_values = [persistence_rmse, rf_rmse]
colors_bars = ['lightcoral', 'lightgreen']
bars = ax9_5.bar(methods, rmse_values, color=colors_bars, edgecolor='black', linewidth=1.5)
ax9_5.set_title('Forecast Model Comparison (RMSE)', fontweight='bold')
ax9_5.set_ylabel('RMSE (C)')
ax9_5.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, rmse_values):
    height = bar.get_height()
    ax9_5.text(bar.get_x() + bar.get_width()/2., height,
               f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

ax9_6 = fig9.add_subplot(gs[2, 2])
ax9_6.axis('off')
summary_text = f"""
KEY METRICS SUMMARY

Total Records: {len(df):,}
Active Operation: {len(df_active):,}

Shutdowns: {num_shutdowns}
Avg Duration: {avg_shutdown_duration:.1f}h

Operating States: {optimal_k}
Anomalies Detected: {len(anomalies_df)}

Forecast RMSE: {rf_rmse:.2f}C
Improvement: {improvement:.1f}%
"""
ax9_6.text(0.1, 0.5, summary_text, fontsize=11, verticalalignment='center',
        family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('CYCLONE SENSOR DATA: COMPREHENSIVE ANALYSIS SUMMARY',
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('plots/09_comprehensive_summary.png', dpi=300, bbox_inches='tight')
plt.close(fig9)
print(f"  -> Saved: plots/09_comprehensive_summary.png")


[6] INSIGHTS & STORYTELLING
--------------------------------------------------------------------------------

* KEY INSIGHTS FROM ANALYSIS:

  1. OPERATIONAL AVAILABILITY
    - Machine availability: 92.1%
    - 416 shutdowns over 3 years (~138 per year)
    - Average shutdown: 6.0 hours
    -> ACTION: Investigate shutdowns exceeding 9h for potential maintenance opportunities.

  2. OPERATING STATE PATTERNS
    - Normal Operation: 3.7% of active time
    - High Load: 78.0% of active time
    - Startup/Low Load: 14.2% of active time
    - Degraded: 4.1% of active time
    -> Dominant state: High Load (78.0%)
    -> ACTION: Focus optimization efforts on the Normal and High Load states.

  3. ANOMALY CONCENTRATION BY STATE
    - Degraded: 15 anomalies (30.0%)
    - High Load: 14 anomalies (28.0%)
    - Normal Operation: 8 anomalies (16.0%)
    - Startup/Low Load: 13 anomalies (26.0%)
    -> Highest risk state for anomalies: Degraded
    -> ACTION: Implement state-specific monitoring thres

In [14]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print("\n* Outputs Generated:")
print("  CSV Files:")
print("    - outputs/shutdown_periods.csv")
print("    - outputs/anomalous_periods.csv")
print("    - outputs/clusters_summary.csv")
print("    - outputs/forecasts.csv")
print("\n  Visualizations:")
print("    - plots/01_correlation_matrix.png")
print("    - plots/02_one_week_view.png")
print("    - plots/03_one_year_view.png")
print("    - plots/04_shutdowns_one_year.png")
print("    - plots/05_clustering_optimization.png")
print("    - plots/06_clusters_visualization.png")
print("    - plots/07_anomaly_*_context.png (3 files)")
print("    - plots/08_forecasting_results.png")
print("    - plots/09_comprehensive_summary.png")

print("\n* All tasks completed successfully!")
print("=" * 80)


ANALYSIS COMPLETE

* Outputs Generated:
  CSV Files:
    - outputs/shutdown_periods.csv
    - outputs/anomalous_periods.csv
    - outputs/clusters_summary.csv
    - outputs/forecasts.csv

  Visualizations:
    - plots/01_correlation_matrix.png
    - plots/02_one_week_view.png
    - plots/03_one_year_view.png
    - plots/04_shutdowns_one_year.png
    - plots/05_clustering_optimization.png
    - plots/06_clusters_visualization.png
    - plots/07_anomaly_*_context.png (3 files)
    - plots/08_forecasting_results.png
    - plots/09_comprehensive_summary.png

* All tasks completed successfully!
